In [ ]:
import numpy as np
import pandas as pd
from labdata.schema import Dataset, DatasetEvents, SpikeSorting, UnitCount
from chipmunk import Chipmunk

In [ ]:
sess = (Chipmunk() & "setting_modalities = 'visual'") & SpikeSorting().fetch(
    "subject_name", "session_name", as_dict=True
)

neural_sess = (
    (Dataset() & "dataset_name LIKE 'ephys%'")
    & SpikeSorting().fetch("subject_name", "session_name", as_dict=True)
    & sess.fetch("subject_name", "session_name", as_dict=True)
)

behavior_data = pd.DataFrame(
    (
        sess.proj()
        * Chipmunk.Trial().proj("response", "rewarded")
        * Chipmunk.TrialParameters().proj("stim_rate_vision")
    ).fetch(as_dict=True)
)

neural_data = pd.DataFrame(
    (
        (
            SpikeSorting().Unit()
            & neural_sess.fetch("subject_name", "session_name", as_dict=True)
        )
        * (UnitCount.Unit() & "unit_criteria_id = 1" & "passes = 1")
    ).fetch(
        "subject_name",
        "session_name",
        "dataset_name",
        "unit_id",
        "spike_times",
        as_dict=True,
    )
)

In [ ]:
z = []
for key in neural_sess:
    events = DatasetEvents.Digital() & key
    if len(events) > 1:
        events = pd.DataFrame(events.fetch_synced())
        align_ev = {
            "stim": events.query("event_name == '0'").event_timestamps.values[0],
            "trial_start": events.query("event_name == '2'").event_timestamps.values[0],
            "left_port": events.query("event_name == '4'").event_timestamps.values[0],
            "center_port": events.query("event_name == '5'").event_timestamps.values[0],
            "right_port": events.query(
                "event_name == '6' & stream_name == 'obx'"
            ).event_timestamps.values[0],
        }

        stim = align_ev["stim"]
        stim_ev = np.concatenate([[stim[0]], stim[1:][np.diff(stim) > 0.025]])
        first_stim_ev = np.concatenate([[stim[0]], stim[1:][np.diff(stim) > 1]])
        align_ev.update({"stim_ev": stim_ev, "first_stim_ev": first_stim_ev})
        z.append(
            {
                "subject_name": key["subject_name"],
                "session_name": key["session_name"],
                **align_ev,
            }
        )

obx_data = pd.DataFrame(z)

In [ ]:
from pathlib import Path
from typing import cast

out_dir = Path("exports/data_export")
out_dir.mkdir(parents=True, exist_ok=True)

exclude_subjects = {"GRB006"}

dfs = {
    "behavior_data": behavior_data,
    "neural_data": neural_data,
    "obx_data": obx_data,
}

for name, df in dfs.items():
    export_df = df.copy()
    if "subject_name" in export_df.columns:
        export_df = export_df[~export_df["subject_name"].isin(exclude_subjects)]

    if {"subject_name", "session_name"}.issubset(export_df.columns):
        for grp_key, sub_df in export_df.groupby(
            ["subject_name", "session_name"], dropna=False
        ):
            subject, session = cast(tuple[str, str], grp_key)
            file_path = out_dir / f"{name}_{subject}_{session}.csv"
            sub_df.to_csv(file_path, index=False)
    else:
        file_path = out_dir / f"{name}.csv"
        export_df.to_csv(file_path, index=False)

print(f"Saved CSV files to: {out_dir} (excluded: {', '.join(exclude_subjects)})")